# Lost in Transcription — Spanish–Nahuatl

Dedicated Spanish–Nahuatl RunPod workflow. Run top-to-bottom: install the compatible GPU stack, download the public training corpus, select by speaker-disjoint competition-dev WER, final-fit, package, then publish the final weights and `submission.zip`. The final competition archive never needs network access.

## 0. RunPod setup (run first on a GPU pod)

This installs the complete Qwen3-ASR, LoRA, packaging, and Hugging Face environment on a fresh Python 3.12 CUDA pod. It is development-only and is not included in the submission archive.

In [ ]:
!pip install -r requirements-dev.txt

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_project_root() -> Path:
    configured = os.environ.get('LIT_ROOT')
    candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if all((candidate / item).exists() for item in ('requirements-runpod.txt', 'src', 'spanish_nahuatl')):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the project root; start Jupyter inside the repository or set LIT_ROOT.')

ROOT = find_project_root()

# Repair a Debian-managed Blinker edge case before qwen-asr resolves Gradio.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--ignore-installed', 'blinker'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--upgrade-strategy', 'eager',
    '-r', str(ROOT / 'requirements-runpod.txt'),
], check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'verify_runpod.py')], check=True)
print('RunPod environment verified. Restart the kernel if pip upgraded an imported package.')


## 1. Configure paths and credentials

Set both data directories. `MDC_API_KEY` is used solely to download Tetelancingo; neither it nor the Hugging Face token is stored in any artifact.

In [ ]:
import json
from getpass import getpass

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SPNH_DATA = ROOT / 'data' / 'spanish_nahuatl'
SPNH_DEV_DIR = SPNH_DATA / 'development'  # metadata.tsv + clips/
TETEL_DIR = SPNH_DATA / 'tetelancingo'
ARTIFACTS = ROOT / 'artifacts' / 'spnh'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
TENSORBOARD_ROOT = ARTIFACTS / 'tensorboard'
TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)

# Preferred: set MDC_API_KEY as a RunPod secret.
if not os.environ.get('MDC_API_KEY'):
    os.environ['MDC_API_KEY'] = getpass('MDC API key (not saved): ')
if not os.environ['MDC_API_KEY'].strip():
    raise RuntimeError('MDC_API_KEY is required to download Tetelancingo')


## 2. Download Tetelancingo

Skip only when `TETEL_DIR` already contains the extracted corpus. The API key must never be copied into the final ZIP.

In [ ]:
if not (TETEL_DIR.exists() and any(TETEL_DIR.rglob('*'))):
    subprocess.run([
        sys.executable, str(ROOT / 'spanish_nahuatl' / 'download_tetelancingo.py'), '--download',
    ], check=True, env=os.environ.copy())
else:
    print(f'Reusing {TETEL_DIR}')


## 3. Build manifests and a held-out speaker split

Use speaker-disjoint competition-dev WER for every selection decision. Tetelancingo’s normalized Nahuatl `sentence` is supervision; its Spanish translation is not.

In [ ]:
from spanish_nahuatl import prepare
from lit_jv import prompts
from lit_jv.decode import DecodeConfig, FallbackConfig, Qwen3AsrDecoder, build_backend
from lit_jv.scoring import wer_report
from lit_jv.train_lora import AugmentSpec, LoraSpec, StageSpec, TrainSpec, merge_and_save, train

dev_rows = prepare.load_competition_dev(SPNH_DEV_DIR)
dev_train, dev_eval = prepare.holdout_speakers(dev_rows, count=1, seed=2026)
tetel_train = prepare.load_tetelancingo(TETEL_DIR, split='train')
tetel_eval = prepare.load_tetelancingo(TETEL_DIR, split='test')
fit_rows = tetel_train + dev_train
terms = prepare.glossary(fit_rows)
(ARTIFACTS / 'glossary.json').write_text(json.dumps(terms, ensure_ascii=False, indent=2), encoding='utf-8')
{'dev_train': len(dev_train), 'dev_eval': len(dev_eval), 'tetel_train': len(tetel_train), 'tetel_eval': len(tetel_eval), 'glossary': len(terms), 'orthography_flags': prepare.orthography_audit(tetel_train)}


## 4. Zero-shot gate

Record this WER. Reject a fine-tune that does not beat it.

In [ ]:
def score_rows(model_path, rows, prompt, name, chunk_size=16):
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(backend, fallback=FallbackConfig(enabled=False), log=print)
    try:
        hypotheses = decoder.decode_pass(rows, DecodeConfig(prompt=prompt, name=name, temperature=0.0), chunk_size=chunk_size)
        report = wer_report([row['raw_text'] for row in rows], [hyp.text for hyp in hypotheses])
        print(report.as_row(name))
        return report
    finally:
        decoder.close()

baseline_prompt = prompts.PromptConfig(prose='codeswitch_spnh', style_tag=prompts.STYLE_VOICE_NOTE_SP_NH, glossary=tuple(terms), max_glossary_terms=128, force_language='Spanish')
baseline = score_rows('Qwen/Qwen3-ASR-1.7B', dev_eval, baseline_prompt, 'qwen-zero-shot')


## 4.5 Multi-pass prompts, error taxonomy, and post-processing

Ported from the Indonesian-Javanese track, which found this combination worth several WER
points before any acoustic change: (1) decode a few deliberately decorrelated prompt
variants instead of one, so ROVER/MBR later has something real to vote over; (2) turn the
truncation-rescue fallback back on (`score_rows` above ships it disabled); (3) fit a
forward-selected post-processing pipeline that only keeps a rule if it lowers WER.

Caveat worth stating plainly: this dev set gives one held-out-speaker split, not the
multi-conversation fold the JV track validates its rules against, so there is no
independent check available beyond `dev_eval` itself here. Treat the rules below as
provisional until §5.5 re-fits them on the fine-tuned model's own hypotheses.

In [ ]:
from lit_jv import combine, postprocess
from collections import Counter

# --- multi-pass prompt design -----------------------------------------
PASS_SPECS = [
    ("p1-glossary", baseline_prompt, 0.0),
    (
        "p2-noglossary",
        prompts.PromptConfig(prose='codeswitch_spnh', style_tag=prompts.STYLE_VOICE_NOTE_SP_NH, glossary=(),
                              force_language='Spanish'),
        0.0,
    ),
    (
        "p3-freelid",
        prompts.PromptConfig(prose='codeswitch_spnh', style_tag=prompts.STYLE_VOICE_NOTE_SP_NH, glossary=tuple(terms[:32]),
                              force_language=None),
        0.0,
    ),
]

# --- length prior + fallback-enabled decode helper ----------------------
HAVE_DURATIONS = bool(dev_eval) and dev_eval[0].get('duration') is not None
if HAVE_DURATIONS:
    PRIOR = combine.LengthPrior.fit(
        [r['raw_text'] for r in dev_train], [r['duration'] for r in dev_train]
    )
    print(f"length prior: mean={PRIOR.mean:.3f} std={PRIOR.std:.3f} words/s")
else:
    PRIOR = None
    print("no 'duration' field on these rows -- fallback ladder / length-prior gating "
          "skipped. Check the field name in prepare.py if that's unexpected.")


def decode_multi_pass(model_path, rows, specs=PASS_SPECS, tag='pass'):
    """Decode `rows` under every spec in `specs`. Returns {name: [text, ...]}."""
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(
        backend, fallback=FallbackConfig(enabled=HAVE_DURATIONS), length_prior=PRIOR, log=print
    )
    out = {}
    try:
        for name, cfg, temperature in specs:
            hyps = decoder.decode_pass(
                rows, DecodeConfig(prompt=cfg, name=f'{tag}-{name}', temperature=temperature),
                chunk_size=16,
            )
            out[name] = [h.text for h in hyps]
    finally:
        decoder.close()
    return out


# --- decode every pass zero-shot, pick the best single one --------------
ZERO_SHOT_PASSES = decode_multi_pass('Qwen/Qwen3-ASR-1.7B', dev_eval, tag='zeroshot')
refs = [r['raw_text'] for r in dev_eval]
zero_shot_wer = {name: round(wer_report(refs, texts).wer, 4) for name, texts in ZERO_SHOT_PASSES.items()}
for name, wer in sorted(zero_shot_wer.items(), key=lambda kv: kv[1]):
    print(f"{name:16s} WER {wer}")
ZS_BEST_PASS = min(zero_shot_wer, key=zero_shot_wer.get)
print('best zero-shot pass:', ZS_BEST_PASS)

# --- error taxonomy on the best pass -------------------------------------
def classify(ref_token, hyp_token):
    if not ref_token:
        return 'insertion'
    if not hyp_token:
        return 'deletion'
    if ref_token == hyp_token:
        return 'identical (alignment artefact)'
    if ref_token.lower() == hyp_token.lower():
        return 'casing only'
    strip = postprocess.strip_accents
    if strip(ref_token).lower() == strip(hyp_token).lower():
        return 'accent only'
    if ref_token.replace('-', '') == hyp_token.replace('-', ''):
        return 'hyphen'
    if strip(ref_token.replace('-', '')).lower() == strip(hyp_token.replace('-', '')).lower():
        return 'mixed style (casing+accent+hyphen)'
    return 'lexical / acoustic'

pairs = [
    (rt, ht)
    for ref, hyp in zip(refs, ZERO_SHOT_PASSES[ZS_BEST_PASS])
    for rt, ht in zip(ref.split(), hyp.split())
]
tax = Counter(classify(a, b) for a, b in pairs if a != b)
total = sum(tax.values()) or 1
for k, v in tax.most_common():
    print(f"{k:35s} {v:5d}  ({v / total:.1%})")

# --- forward-selected post-processing, gated on WER ----------------------
LEXICON = postprocess.StyleLexicon.fit(refs, min_count=2, dominance=0.9)
PIPELINE = postprocess.RulePipeline(lexicon=LEXICON).fit(
    refs, ZERO_SHOT_PASSES[ZS_BEST_PASS], verbose=True
)
_durations = [r['duration'] for r in dev_eval] if HAVE_DURATIONS else [0.0] * len(dev_eval)
_repaired = PIPELINE.apply_many(ZERO_SHOT_PASSES[ZS_BEST_PASS], _durations)
print(f"{ZS_BEST_PASS} + post-processing: {wer_report(refs, _repaired).wer:.4f} "
      f"(zero-shot alone: {zero_shot_wer[ZS_BEST_PASS]:.4f})")


## 5. Train the selected recipe

Stage 1 learns Nahuatl vocabulary/acoustics from Tetelancingo; stage 2 prioritizes Spanish–Nahuatl voice-note style. Generation WER selects checkpoints; teacher-forced evaluation remains disabled.

In [ ]:
spec = TrainSpec(
    model_path='Qwen/Qwen3-ASR-1.7B', output_dir=str(ARTIFACTS / 'lora'), target_language='Spanish',
    lora=LoraSpec(r=16, alpha=32, adapt_text_decoder=True, adapt_audio_encoder=False),
    augment=AugmentSpec(corpora=('tetelancingo',)), prompt=baseline_prompt,
    eval_steps=0, wer_eval_steps=10, wer_eval_clips=len(dev_eval), wer_eval_batch_size=4, run_name='spnh-qwen3asr',
)
stages = (
    StageSpec('s1-tetelancingo', {'tetelancingo': 1.0, 'spnh_dev': 1.25}, epochs=2.0, learning_rate=1.0e-4, batch_size=4, grad_accumulation=8),
    StageSpec('s2-voice-note', {'spnh_dev': 1.0, 'tetelancingo': 0.15}, epochs=5.0, learning_rate=3.0e-5, batch_size=4, grad_accumulation=8),
)
selected = train(fit_rows, spec, stages=stages, save_adapter_as='selected', eval_rows=dev_eval)
selected['best_wer']


## 6. Select, final-fit, and package

If the assertion fails, stop and change one variable before rerunning from the baseline. Once it passes, final fitting freezes the recipe and uses every released training row.

In [ ]:
merged = merge_and_save(spec, ARTIFACTS / 'lora' / 'selected', ARTIFACTS / 'merged')
selected_dev = score_rows(merged, dev_eval, spec.prompt, 'selected-dev')
selected_tetel = score_rows(merged, tetel_eval, spec.prompt, 'selected-tetel')
assert selected_dev.wer < baseline.wer, 'Reject this recipe: it did not beat zero-shot Qwen.'


### 5.5 Combine passes on the selection checkpoint (held-out, evidence-based)

Decode `dev_eval` with the **selection** checkpoint (before the final fit rolls it into full
training data) across all three prompt passes, so the pass-count / combiner decision is
measured on this model rather than assumed from the zero-shot correlation. This mirrors the
JV track's own finding that ROVER/MBR are unreliable with only two candidates (a 1-1 tie
resolved by tie-break) — so `none` is a legitimate winner here, not a fallback of last
resort.

In [ ]:
SEL_PASSES = decode_multi_pass(merged, dev_eval, tag='selected')
sel_wer = {name: round(wer_report(refs, texts).wer, 4) for name, texts in SEL_PASSES.items()}
for name, wer in sorted(sel_wer.items(), key=lambda kv: kv[1]):
    print(f"{name:16s} WER {wer}")
SEL_BEST_PASS = min(sel_wer, key=sel_wer.get)

# Refit post-processing on this checkpoint's own mistakes.
SEL_LEXICON = postprocess.StyleLexicon.fit(refs, min_count=2, dominance=0.9)
SEL_PIPELINE = postprocess.RulePipeline(lexicon=SEL_LEXICON).fit(
    refs, SEL_PASSES[SEL_BEST_PASS], verbose=True
)

# Measure ROVER / MBR at every surviving pass count -- do not assume voting helps.
combiner_by_n = {}
sel_names = list(SEL_PASSES)
for n in range(1, len(sel_names) + 1):
    subset = sel_names[:n]
    single = wer_report(refs, SEL_PIPELINE.apply_many(SEL_PASSES[subset[0]], _durations)).wer
    best_label, best_wer = "none", single
    if n >= 2:
        for label, fn in (("mbr", combine.mbr_select), ("rover", combine.rover_combine)):
            texts = []
            for i in range(len(dev_eval)):
                candidates = [SEL_PASSES[m][i] for m in subset]
                texts.append(fn(candidates).text)
            wer = wer_report(refs, SEL_PIPELINE.apply_many(texts, _durations)).wer
            if wer < best_wer:
                best_label, best_wer = label, wer
    combiner_by_n[n] = best_label
    print(f"{n} pass(es): single={single:.4f}  selected={best_label} ({best_wer:.4f})")

SUBMISSION_COMBINER = combiner_by_n[len(sel_names)]
if SUBMISSION_COMBINER == "none":
    SUBMISSION_SPECS = [next(s for s in PASS_SPECS if s[0] == SEL_BEST_PASS)]
    print(f"shipping a single pass ({SEL_BEST_PASS}); combining did not help on this checkpoint.")
else:
    SUBMISSION_SPECS = PASS_SPECS
    print(f"shipping all {len(sel_names)} passes with combiner={SUBMISSION_COMBINER!r}.")


In [ ]:
final_rows = prepare.load_tetelancingo(TETEL_DIR, split=None) + dev_rows
final_spec = TrainSpec(**{**spec.__dict__, 'output_dir': str(ARTIFACTS / 'final-lora'), 'wer_eval_steps': 0})
train(final_rows, final_spec, stages=stages, save_adapter_as='final', eval_rows=None)
final_merged = merge_and_save(final_spec, ARTIFACTS / 'final-lora' / 'final', ARTIFACTS / 'final-merged')
package = subprocess.run([
    sys.executable, str(ROOT / 'spanish_nahuatl' / 'package_submission.py'),
    '--model', str(final_merged), '--glossary', str(ARTIFACTS / 'glossary.json'),
], check=True, capture_output=True, text=True)
print(package.stdout)
SUBMISSION_ZIP = ARTIFACTS / 'submission.zip'
assert SUBMISSION_ZIP.is_file() and SUBMISSION_ZIP.stat().st_size > 0
print(f'Ready to publish: {final_merged} and {SUBMISSION_ZIP}')


### 6.5 Refit post-processing on the final production checkpoint

`final_merged` is trained on *all* released dev rows (§6 above), so `dev_eval` is no longer
held out from it -- this refit is in-sample by construction, exactly like the JV track's own
final pipeline (its §8 says as much: "fitted on the whole development set, saved next to
main.py"). The held-out check that matters already happened in §5.5, on the selection
checkpoint. This step's only job is to make sure the shipped rules reflect the *actual*
production model's error distribution rather than the selection checkpoint's.

**Before packaging**, confirm `package_submission.py` for this track actually embeds
`postprocess.json` / a multi-pass + combiner config into the archive the way the JV track's
`submission_config.json` does — the call below only passes `--model` and `--glossary` today.
If the script doesn't yet accept a rules/passes argument, either extend it (mirroring the JV
track's `CONFIG` dict in its packaging section) or copy `postprocess.json` into
`submission_src/` by hand before zipping, and check that `main.py` actually loads it.

In [ ]:
FINAL_PASSES = decode_multi_pass(final_merged, dev_rows, specs=SUBMISSION_SPECS, tag='final')
final_refs = [r['raw_text'] for r in dev_rows]
final_lexicon = postprocess.StyleLexicon.fit(final_refs, min_count=2, dominance=0.9)
FINAL_PIPELINE = postprocess.RulePipeline(lexicon=final_lexicon).fit(
    final_refs, FINAL_PASSES[SUBMISSION_SPECS[0][0]], verbose=True
)
FINAL_PIPELINE.save(ARTIFACTS / 'postprocess.json')
print(f"saved {len(FINAL_PIPELINE.rules)} refit rules to {ARTIFACTS / 'postprocess.json'}")
print(f"shipped config: passes={[s[0] for s in SUBMISSION_SPECS]} combiner={SUBMISSION_COMBINER!r}")
print("TODO: verify package_submission.py embeds postprocess.json and this pass/combiner "
      "config into submission.zip before relying on it in the real container.")


## 7. Publish final model weights and `submission.zip` to Hugging Face

Set `SPNH_HF_REPO` to an `owner/repository` destination if desired. Otherwise, this cell creates `lost-in-transcription-spanish-nahuatl` under the authenticated account. `HF_TOKEN` is requested privately if it is not already a RunPod secret; it is never saved to a model, ZIP, or notebook output.

In [ ]:
from huggingface_hub import HfApi, hf_hub_url

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Hugging Face write token (not saved): ')
if not os.environ['HF_TOKEN'].strip():
    raise RuntimeError('HF_TOKEN is required to publish the model and submission archive')

api = HfApi(token=os.environ['HF_TOKEN'])
account = api.whoami()['name']
REPO_ID = os.environ.get('SPNH_HF_REPO', f'{account}/lost-in-transcription-spanish-nahuatl')
PRIVATE = os.environ.get('SPNH_HF_PRIVATE', '0') == '1'
api.create_repo(repo_id=REPO_ID, repo_type='model', private=PRIVATE, exist_ok=True)

# Publish final-merged exactly once under model/, and the competition archive at the root.
model_commit = api.upload_folder(
    repo_id=REPO_ID, repo_type='model', folder_path=str(final_merged), path_in_repo='model',
    commit_message='Upload final Spanish-Nahuatl merged model weights',
    ignore_patterns=['**/__pycache__/**', '*.lock'],
)
archive_commit = api.upload_file(
    path_or_fileobj=str(SUBMISSION_ZIP), path_in_repo='submission.zip', repo_id=REPO_ID, repo_type='model',
    commit_message='Upload Spanish-Nahuatl competition submission archive',
)
api.upload_file(
    path_or_fileobj=str(ARTIFACTS / 'glossary.json'), path_in_repo='assets/glossary.json', repo_id=REPO_ID, repo_type='model',
    commit_message='Upload final glossary provenance',
)
print('Model weights:', getattr(model_commit, 'commit_url', model_commit))
print('Submission archive:', getattr(archive_commit, 'commit_url', archive_commit))
print('Repository:', f'https://huggingface.co/{REPO_ID}')
print('Archive download:', hf_hub_url(repo_id=REPO_ID, filename='submission.zip', repo_type='model'))
